# Pure $\zeta(3)$: divisor certificates and $\mu(\zeta(3))<5.4197$

This notebook independently reproduces the arithmetic of **Appendix B.4** (`app:zeta3-measure`) and the pure-$\zeta(3)$ conclusion of **Theorem 6.16 (Irrationality measures)** (`thm:z23-measures`). The error-rate input is **Lemma 6.8** (`lem:z34-selected-error`), and the limit input is **Lemma 6.7** (`lem:z34-limits`). Appendix B.4 contains named paragraphs rather than numbered individual lemmas; we do not invent lemma numbers for the carry, shell, or trace steps.

Run all cells in a fresh kernel. No prior notebook execution is needed. The accompanying Python modules and exact reference data must remain in this folder. No Lean code is written or executed. The supplied formalization is treated as given background.

**Three different kinds of evidence.** (1) The 270-cell computations below are exhaustive exact rational arrangement certificates. (2) The finite $n$ checks are only regression tests. (3) Turning the certificates into a uniform divisor and asymptotic rate uses the paper's written carry, shell, trace, readout, and prime number theorem arguments. Those analytic and uniform statements are not inferred from finite gcd experiments.


### Identification used in this conclusion

The appendix sums satisfy
$C'_n(Q_n,P_{3,n})=(-1)^n(\mathbf q_n,\mathbf p_{3,n})$ at the same index $n$.
For completeness, the reason is the meromorphic identity
\[
C'_nF_n(-2n-z)=(-1)^n\frac{\pi^2}{\sin^2(\pi z)}
\prod_{u=2}^{4}\binom{z+(10+u)n}{10n}\binom{15n}{n+z}\binom{15n}{z}.
\]
Comparing its double- and simple-pole coefficients at integer $z=k$ gives
$C'_na_{2n+k}=(-1)^nB_{n,k}$ and $C'_nb_{2n+k}=-(-1)^nB'_{n,k}$.
Since $j+2n=k+4n$, summation gives the stated identity, including the simple-pole tails.
Hence $P_{3,n}/Q_n=p_{3,n}/q_n$, so the Section 6 error estimate applies to the arithmetic normalization used here. The full explanation is also in notebook 01; no execution of that notebook is needed.

In [1]:
from pathlib import Path
import sys, time, json, math
# Supports opening this notebook directly or running from the bundle root.
HERE = Path.cwd()
if not (HERE / 'approximants.py').exists():
    HERE = HERE / 'pure_zeta3'
assert (HERE / 'approximants.py').exists(), 'Run from this folder or the bundle root.'
sys.path.insert(0, str(HERE.resolve()))
if hasattr(sys, 'set_int_max_str_digits'):
    sys.set_int_max_str_digits(0)
started = time.perf_counter()
from fractions import Fraction as F
from flint import fmpq, arb, ctx
import sympy as sp
from approximants import partial_fractions, partial_fractions_direct, kernel_readout, finite_integer_check
from rates import native_trajectory, matrix_at_infinity, spectral_enclosures


## 1. Appendix B.4: baseline carries on the Farey-29 partition

For a prime with $p^2>30n$, Legendre's formula gives factorial valuations using only $\lfloor m/p\rfloor$. Put $x=n/p$ and $y=j/p$. For a double pole $2n\le j\le16n$, the valuation of $C'_na_j$ is
\[
s(x,y)=2\lfloor15x\rfloor-3\lfloor10x\rfloor
+\sum_{u=0}^2\bigl(\lfloor y+(10+u)x\rfloor-\lfloor y+ux\rfloor\bigr)
-\lfloor y-x\rfloor-\lfloor y-2x\rfloor-\lfloor16x-y\rfloor-\lfloor17x-y\rfloor.
\]
The logarithmic derivative giving $b_j$ can lose one prime, except when its exact harmonic coefficient cancels. The `coefficient_valuation_bounds` function also handles the simple-pole ranges $n\le j<2n$ and $16n<j\le17n$, the $d_{19n}$ factor, and whether $j+2n\ge p$.

The profile is the minimum over all possible terms. All changes occur along the affine boundaries listed in `chambers.py`; their abscissa crossings lie on the Farey partition of order 29. Every open cell and every relevant $y$-boundary stratum is checked using exact fractions. For $x>1$, the baseline repeats using the separate `tail` value; the first-period `g3` must not be repeated blindly. A midpoint represents a whole open chamber because no affine boundaries cross inside it. The finite list of exceptional walls cannot contain $\{n/p\}$ for $p>29$, $p\nmid n$.


In [2]:
from chambers import build_chambers, compare_reference, merge_runs
from rates import density_enclosures, final_bounds
checkpoint = time.perf_counter()
rows = build_chambers()
assert compare_reference(rows)
assert len(rows) == 270
print('Regenerated 270 baseline/first-order/second-order cells.')
print('Every regenerated exact certificate agrees with the supplied reference.')
print('Chamber runtime (seconds):', round(time.perf_counter()-checkpoint, 2))


Regenerated 270 baseline/first-order/second-order cells.
Every regenerated exact certificate agrees with the supplied reference.
Chamber runtime (seconds): 4.43


## 2. Appendix B.4: fourteen first-order shells

For residue $k\pmod p$, split the integer roots of the rational kernel into low residues and high roots. If $N_k,D_k$ are the numerator and denominator high-root multisets, let $h_k=|N_k|-|D_k|$. On a first-order shell, all $h_k\ge-1$, and the high-kernel readout is identical on the minimal classes $h_k=-1$. The accompanying finite-field summation argument gives one extra common prime factor.

`first_order_cell` checks these high-root signatures and exact rational readout constants; merging its valid cells produces the fourteen open intervals below. Each interval applies to every periodic lift $\{n/p\}$, not only to $0<n/p<1$. The two second-order windows are a further gain inside these shells, so their indicators are added, not substituted.


In [3]:
first_order_bands = merge_runs(rows)
assert len(first_order_bands) == 14
for a, b in first_order_bands:
    print(f'({a}, {b})')


(1/11, 1/8)
(3/13, 4/17)
(3/11, 5/18)
(4/13, 5/16)
(4/11, 7/19)
(5/13, 7/18)
(5/12, 8/19)
(5/11, 8/17)
(6/11, 9/16)
(7/12, 10/17)
(7/11, 11/17)
(7/10, 12/17)
(8/11, 11/15)
(10/11, 16/17)


## 3. Appendix B.4: the two second-order windows

The written trace lemma assumes $h_k\ge-1$, a common high-root function $U(V)$ on every minimal class, and $U_k(V)=(V-a_k)U(V)$ for each class with $h_k=0$. Its conclusion is
\[
p\sum_{k\bmod p}F_n(k+pV)\equiv0\pmod{p^2}.
\]
Together with the readout argument this yields $v_p(Q_n)\ge3$ and $v_p(P_{3,n})\ge0$ in the two windows. It contributes one additional factor after the first-order gain.

`second_order_cell` verifies the hypotheses through **affine endpoint identities**. An endpoint is stored as a pair $(m,c)$ representing $mt+c$, where $t=\lfloor n/p\rfloor$. Subtracting signed interval boundaries leaves a single linear factor, identically in $t$. Thus these are symbolic checks of all periodic lifts, not experiments at selected integer lifts. The residue-zero condition and high-pole separation are part of the written trace application; the certificate checks the residue-zero multiplicity, while $p>29$ and $p^2>30n$ give the required unit differences.


In [4]:
second_order_bands = merge_runs(rows, second=True)
assert second_order_bands == [(F(1,10), F(2,19)), (F(11,12), F(15,16))]
assert sum(row['second']['valid'] for row in rows) == 9
print('Nine valid open Farey cells merge into:', second_order_bands)
print('Total checked cells and y-boundary strata:', sum(row['point_count'] for row in rows))


Nine valid open Farey cells merge into: [(Fraction(1, 10), Fraction(2, 19)), (Fraction(11, 12), Fraction(15, 16))]
Total checked cells and y-boundary strata: 5400


## 4. Appendix B.4: the divisor density, with an infinite-tail bound

The prime number theorem converts the uniform exponent profile $\gamma(x)$ to
\[
\liminf_{n\to\infty}\frac1n\log\mathcal D'_n
\ge\int_0^\infty\gamma(x)x^{-2}\,dx.
\]
The exclusions $p\le29$, $p^2\le30n$, and $p\mid n$ have negligible exponential cost in the written argument.

A periodic interval $(a,b)$ contributes $\sum_{t\ge0}(1/(t+a)-1/(t+b))$. We sum 1000 periods, rounding **downward** at $10^{-18}$. For the remaining convex positive tail $f(t)=1/(t+a)-1/(t+b)$,
\[
\log\frac{K+b}{K+a}+\frac{f(K)}2
\le\sum_{t\ge K}f(t)
\le\log\frac{K-\tfrac12+b}{K-\tfrac12+a}.
\]
The logarithms themselves have rational atanh-series bounds. There is no uncontrolled numerical quadrature. The resulting lower endpoint is exactly the rational printed in Appendix B.4.


In [5]:
density = density_enclosures(rows)
for label, (lo, hi) in density.items():
    print(f'{label:14s}: [{float(lo):.15f}, {float(hi):.15f}]')
assert density['total'][0] == F(30458547243805493774844991, 10**24)
assert density['total'][0] > F('30.458547')
print('Exact lower endpoint:', density['total'][0])


baseline      : [26.237825155363886, 26.237825155654004]
first_order   : [3.674882827451926, 3.674882827490592]
second_order  : [0.545839260989681, 0.545839260996196]
total         : [30.458547243805494, 30.458547244140792]
Exact lower endpoint: 30458547243805493774844991/1000000000000000000000000


## 5. Theorem 6.16: final height and irrationality measure

The base integers are $6C'_nd_{19n}^3(Q_n,P_{3,n})$. The factor $d_{19n}^3$ contributes $57$ to the exponential rate. After division by $\mathcal D'_n$,
\[
h\le\log\lambda+30\log(3/2)+57-\gamma.
\]
The spectral fixture is an exact rational matrix/polynomial input, independently reconstructed from `ramanujantools` in Notebook 01. This notebook rechecks its cubic root bracket, discriminant, and logarithm bounds without depending on Notebook 01's execution.

**Decimal correction in the manuscript.** $30\log(3/2)=12.163953243\ldots$, so the printed $12.163953$ is not an upper bound by itself. The rigorous combined bound is $\log\lambda+30\log(3/2)<65.451536565866$. With the certified divisor rate it gives $h<91.992989322061$, hence the paper's stated $h<91.992990$ still holds. The code uses this correct enclosure.

Given Lemmas 6.7 and 6.8 and the paper's nonvanishing/irrationality-measure argument,
\[
g>112.8075,\quad h<91.992990,\quad
\delta=\frac gh-1>0.2262,\qquad
\mu(\zeta(3))\le\frac g{g-h}<5.4197.
\]
Here $g/(g-h)$ decreases with $g$ and increases with $h$, so the conservative lower $g$ and upper $h$ give the required upper bound.


In [6]:
spectral = spectral_enclosures()
bounds = final_bounds(spectral, density)
for label, value in bounds.items():
    print(f'{label:22s}: {float(value):.12f}')
assert bounds['g_lower'] == F('112.8075')
assert bounds['h_upper'] == F('91.992990')
assert bounds['delta_lower'] > F('0.2262')
assert bounds['mu_upper'] < F('5.4197')


g_lower               : 112.807500000000
h_upper               : 91.992990000000
delta_lower           : 0.226261914087
mu_upper              : 5.419656768283
sharper_delta_lower   : 0.226262349759


## 6. Separate finite regression checks

These examples verify the integer representatives and divisibility by the actual reduced denominator. They are intentionally separate from the exact profile and infinite-density proof above. The first/second-order prime counts show that the added arithmetic is actually exercised.


In [7]:
for n in (10, 49, 99):
    print(finite_integer_check(n, rows))
print('All exact certificate and final-bound assertions passed.')
print('Total notebook runtime (seconds):', round(time.perf_counter()-started, 2))


{'n': 10, 'first_order_primes': 8, 'second_order_primes': 1, 'integer_denominator_digits': 415, 'reduced_denominator_digits': 392}


{'n': 49, 'first_order_primes': 30, 'second_order_primes': 4, 'integer_denominator_digits': 2000, 'reduced_denominator_digits': 1955}


{'n': 99, 'first_order_primes': 54, 'second_order_primes': 8, 'integer_denominator_digits': 4004, 'reduced_denominator_digits': 3936}
All exact certificate and final-bound assertions passed.
Total notebook runtime (seconds): 8.66


The archived scripts sometimes retain `NUMERICAL_CANDIDATE` or `conditional` labels from earlier exploratory stages. This notebook does not turn those labels into a new proof claim: it states exactly which certificates are recomputed and which uniform mathematical arguments are inputs. The supplied Lean files are neither modified nor required for these Python runs.
